# 🏥 Cigna 국제 의료보험 RAG Q&A 데모

**SKN24 3차 프로젝트 — 외국인 내담자 보험 약관 Q&A 시스템**

이 노트북은 두 가지 데모를 보여줍니다:
- **Demo 1**: 일반 LLM vs RAG — 카운슬링 세션 보장 관련 질문
- **Demo 2**: 표(Table) 기반 RAG — pdfplumber로 보장 금액 표 추출 후 Q&A

### 사용 모델 / 도구
| 구성 요소 | 선택 |
|---|---|
| LLM | `openai:gpt-4.1-mini` |
| 임베딩 | `text-embedding-3-small` |
| 벡터 DB | Chroma (로컬) |
| PDF 텍스트 | PyPDFLoader |
| PDF 표 추출 | pdfplumber (Gemini 추천) |

### 필수 준비
- `.env` 파일에 `OPENAI_API_KEY=sk-...` 저장
- Cigna PDF 파일을 `PDF_PATH` 변수에 지정

## 0. 패키지 설치

처음 실행 시에만 필요합니다.

In [37]:
# 필요 패키지 설치 (처음 실행 시에만)
# !pip install langchain langchain-community langchain-openai langchain-chroma pdfplumber python-dotenv chromadb -q

## 1. 환경 설정

`.env` 파일 예시:
```
OPENAI_API_KEY=sk-xxxxxxxxxxxxxxxxxxxxxxxx
```

In [38]:
# ──────────────────────────────────────────────────────────────
# 출처: 01_rag_workflow-9e1618d6.ipynb, Cell 1
# ──────────────────────────────────────────────────────────────
import os
from dotenv import load_dotenv

# .env 파일에서 OPENAI_API_KEY 로드
load_dotenv()

api_key = os.getenv('OPENAI_API_KEY')
print(f"✅ API Key 로드: {'성공' if api_key else '실패 → .env 파일 확인 필요'}")

✅ API Key 로드: 성공


## 2. PDF 파일 경로 설정

In [39]:
# PDF 파일 경로를 실제 위치에 맞게 수정하세요
PDF_PATH = '591048 CGHO Customer Guide EN_05_2022.pdf'


if os.path.exists(PDF_PATH):
    print(f'✅ PDF 확인 완료: {PDF_PATH}')
else:
    print(f'❌ PDF 없음: {PDF_PATH}')
    print('   → PDF_PATH 변수를 올바른 경로로 수정해주세요.')

✅ PDF 확인 완료: 591048 CGHO Customer Guide EN_05_2022.pdf


## 3. PDF 텍스트 로드 (PyPDFLoader)

PyPDFLoader는 페이지 단위로 텍스트를 추출하며, 각 Document에 `page` 메타데이터가 자동으로 포함됩니다.

In [40]:
# 1. 기존의 불안정한 패키지 제거 ㅋ
# !pip uninstall transformers trl pypdf -y

# 2. 인텔 맥 최적화 버전으로 재설치 (은우님의 LRScheduler 에러 방지용 ㅋ)
# !pip install "transformers<4.45.0" "trl<0.10.0"

# 3. 에러 없이 Cigna PDF를 읽어줄 PyMuPDF 설치 (이게 핵심! ㅋ)
# !pip install pymupdf

In [41]:
# ──────────────────────────────────────────────────────────────
# 출처: 01_rag_workflow-9e1618d6.ipynb, Cell 4
# ──────────────────────────────────────────────────────────────
from langchain_community.document_loaders import PyPDFLoader
# pypdf 패키지는 매우 가볍지만, 복잡하게 압축된 최신 PDF 형식에는 에러 날 수 있음

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

print(f'📄 총 {len(docs)}개 페이지 로드 완료')
print('\n--- 샘플 (p.36 Health & Wellbeing) ---')
# page 인덱스는 0부터 시작 (실제 페이지 = index + 1)
if len(docs) > 35:
    print(docs[35].page_content[:500])

📄 총 43개 페이지 로드 완료

--- 샘플 (p.36 Health & Wellbeing) ---
36   |   www.cignaglobal.com
INTERNATIONAL HEALTH & WELLBEING 
We understand the importance of your overall wellbeing and living a balanced life. In addition to 
health screenings, tests and examinations; this module also empowers you and your family with the 
services and support to manage your own individual day-to-day health and wellbeing. Your Wellness 
companion, comprising of the Life Management Assistance programme and the Telephonic Wellness 
Coaching, is available to help you and your f


In [42]:
# # 제미나이 코드
# from langchain_community.document_loaders import PyMuPDFLoader
# from langchain_text_splitters import RecursiveCharacterTextSplitter

# # 1. 로더 교체 (에러 방지 ㅋ)
# loader = PyMuPDFLoader(PDF_PATH)
# docs = loader.load()

# # 2. 은우님의 센스 있는 분할 설정 유지 ㅋ
# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=500,
#     chunk_overlap=50,
#     separators=['\n\n', '\n', '. ', ' ', '']
# )

# split_docs = text_splitter.split_documents(docs)

# print(f'✅ 에러 없이 {len(docs)}페이지 로드 완료! ㅋ')
# print(f'🔪 분할 완료: {len(split_docs)}개 청크 생성')

## 4. 텍스트 청크 분할 (RecursiveCharacterTextSplitter)

보험 문서는 문장이 길어 `chunk_size=500`으로 설정합니다 (강사 코드 원본: 200).  
`chunk_overlap`은 청크 간 문맥 연속성을 보장합니다.

In [43]:
# ──────────────────────────────────────────────────────────────
# 출처: 01_rag_workflow-9e1618d6.ipynb, Cell 12
# ──────────────────────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,    # 원본(200)보다 크게 - 보험 약관 문장이 길기 때문
    chunk_overlap=50,  # 원본: 35
    separators=['\n\n', '\n', '. ', ' ', '']
)

split_docs = text_splitter.split_documents(docs)

print(f'🔪 분할 결과: {len(docs)}페이지 → {len(split_docs)}개 청크')
print('\n--- 청크 샘플 ---')
print(split_docs[0].page_content[:300])
print(f'\nMetadata: {split_docs[0].metadata}')

🔪 분할 결과: 43페이지 → 260개 청크

--- 청크 샘플 ---
CUSTOMER GUIDE
Everything you need to know about your plan
Cigna Global Health Options

Metadata: {'producer': 'Adobe PDF Library 16.0.5', 'creator': 'Adobe InDesign 17.1 (Windows)', 'creationdate': '2022-03-28T11:05:13+02:00', 'moddate': '2022-03-28T18:08:15+01:00', 'trapped': '/False', 'source': '591048 CGHO Customer Guide EN_05_2022.pdf', 'total_pages': 43, 'page': 0, 'page_label': '1'}


## 5. 임베딩 + Chroma 벡터 저장소

- 임베딩: `text-embedding-3-small` (강사 코드와 동일)  
- 벡터 DB: **Chroma** (로컬, 별도 서버 불필요)

In [44]:
# ──────────────────────────────────────────────────────────────
# 출처: 01_rag_workflow-9e1618d6.ipynb, Cell 15 (임베딩), Cell 20 (Chroma)
# ──────────────────────────────────────────────────────────────
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_chroma.vectorstores import Chroma

# OpenAI 임베딩 모델 초기화
embedding_model = OpenAIEmbeddings(model='text-embedding-3-small')

# Chroma 벡터 저장소 생성 (Pinecone 대신 로컬 사용)
vector_store = Chroma.from_documents(
    documents=split_docs,
    embedding=embedding_model,
    collection_name='cigna_policy_text'
)

print('✅ Chroma 벡터 저장소 생성 완료')
print(f'   저장된 벡터 수: {vector_store._collection.count()}개')

✅ Chroma 벡터 저장소 생성 완료
   저장된 벡터 수: 520개


## 6. 리트리버(Retriever) 설정

질문과 가장 유사한 청크 **4개**를 검색합니다.

In [45]:
# ──────────────────────────────────────────────────────────────
# 출처: 01_rag_workflow-9e1618d6.ipynb, Cell 22
# ──────────────────────────────────────────────────────────────
retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 6}  # 상위 6개 검색 (한-영 임베딩 매칭 보완)
)

# 동작 확인
test_result = retriever.invoke('counseling sessions life management programme')
print(f'🔍 테스트 검색: {len(test_result)}개 문서 검색됨')
for i, d in enumerate(test_result):
    page_num = d.metadata.get('page', 0)
    print(f'  [{i+1}] p.{page_num+1}: {d.page_content[:80]}...')

🔍 테스트 검색: 6개 문서 검색됨
  [1] p.5: through counselling, telephone support and online programmes.
Life Management As...
  [2] p.5: through counselling, telephone support and online programmes.
Life Management As...
  [3] p.36: Coaching, is available to help you and your family stay healthy and well, both p...
  [4] p.36: Coaching, is available to help you and your family stay healthy and well, both p...
  [5] p.36: experiencing stress, and challenges with focus and concentration.
 › An online s...
  [6] p.36: experiencing stress, and challenges with focus and concentration.
 › An online s...


---
## 🔍 Demo 1: 일반 LLM vs RAG 비교

**질문**: Cigna 보험에서 카운슬링 세션은 몇 회까지 보장되나요?

- **일반 LLM**: 학습 데이터 기반 → Cigna 약관의 정확한 내용을 모를 수 있음
- **RAG**: Cigna PDF에서 관련 내용을 검색한 뒤 답변 → 정확한 약관 내용 기반

In [46]:
# ──────────────────────────────────────────────────────────────
# LLM + 프롬프트 + LCEL 체인 구성
# 출처: 01_rag_workflow-9e1618d6.ipynb, Cell 28-30
#        02-6_contextual_comp_index-b3481113.ipynb (init_chat_model)
# ──────────────────────────────────────────────────────────────
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# GPT-4.1-mini: 성능/비용 균형, 빠른 추론
model = init_chat_model('openai:gpt-4.1-mini', temperature=0.3)

# RAG 프롬프트 템플릿: 문서 내용만 참고, 없으면 모른다고 답변
PROMPT_TEMPLATE = (
    '당신은 Cigna 국제 의료보험 전문 안내원입니다.\n'
    '아래 제공된 보험 약관 문서 내용만을 참고하여 질문에 정확하게 답변하세요.\n'
    '문서에 없는 내용은 "문서에서 확인되지 않습니다"라고 답변하세요.\n'
    '⚠️ 보장 범위, 횟수, 이용 방법, 제약 조건(예: 지정 provider 경유 필수 등) 등\n'
    '모든 관련 세부 사항과 주의사항을 빠짐없이 답변에 포함하세요.\n\n'
    '[참고 문서]\n{context}\n\n'
    '[질문]\n{question}\n\n'
    '[답변]\n'
)

rag_prompt = PromptTemplate.from_template(PROMPT_TEMPLATE)

# 문서 포맷 함수 - 페이지 출처 포함
# 출처: 01_rag_workflow-9e1618d6.ipynb, Cell 28
def format_docs(docs):
    return '\n\n'.join(
        f"[출처: p.{doc.metadata.get('page', 0)+1}]\n{doc.page_content}"
        for doc in docs
    )

# LCEL 체인 구성
# 출처: 01_rag_workflow-9e1618d6.ipynb, Cell 30
rag_chain = (
    {
        'question': RunnablePassthrough(),
        'context': retriever | format_docs
    }
    | rag_prompt
    | model
    | StrOutputParser()
)

print('✅ RAG 체인 구성 완료 (retriever → prompt → gpt-4.1-mini → parser)')

✅ RAG 체인 구성 완료 (retriever → prompt → gpt-4.1-mini → parser)


### Demo 1-A: 일반 LLM 답변 (RAG 없음)

In [51]:
# ──────────────────────────────────────────────────────────────
# 일반 LLM 직접 질문 (문서 검색 없음)
# ──────────────────────────────────────────────────────────────
from langchain_core.messages import HumanMessage

# 🔑 영어 키워드 포함 → 영문 PDF와 임베딩 유사도 매칭 개선
question = (
    'Cigna 국제 의료보험(Global Health Options)에서 '
    'Life Management Assistance Programme의 '
    'counselling 세션은 몇 번(per issue per period of cover)까지 보장되나요? '
    '전화(telephone), 화상(video), 대면(face-to-face) 방식 모두 가능한가요? '
    '이용 시 주의사항(provider 관련)도 함께 알려주세요.'
)

print('=' * 65)
print('❓ 질문:', question)
print('=' * 65)
print()
print('📢 [일반 LLM 답변] — Cigna 약관 문서 없이 GPT-4.1-mini 직접 응답')
print('-' * 65)

# RAG 없이 직접 LLM에 질문
plain_response = model.invoke([HumanMessage(content=question)])
print(plain_response.content)
print()
print('⚠️  주의: 위 답변은 LLM 학습 데이터 기반이며 실제 약관과 다를 수 있습니다.')

❓ 질문: Cigna 국제 의료보험(Global Health Options)에서 Life Management Assistance Programme의 counselling 세션은 몇 번(per issue per period of cover)까지 보장되나요? 전화(telephone), 화상(video), 대면(face-to-face) 방식 모두 가능한가요? 이용 시 주의사항(provider 관련)도 함께 알려주세요.

📢 [일반 LLM 답변] — Cigna 약관 문서 없이 GPT-4.1-mini 직접 응답
-----------------------------------------------------------------
Cigna 국제 의료보험(Global Health Options)에서 제공하는 Life Management Assistance Programme(LMAP)의 상담(counselling) 세션 관련 내용은 다음과 같습니다:

1. **보장 횟수 (per issue per period of cover)**  
   - 일반적으로 LMAP 상담 세션은 **문제(issue) 당 보험 가입 기간(period of cover) 내에 최대 6회**까지 보장됩니다.  
   - 즉, 한 가지 문제에 대해 최대 6회의 상담 세션을 받을 수 있습니다.

2. **상담 방식**  
   - **전화(telephone)** 상담  
   - **화상(video)** 상담  
   - **대면(face-to-face)** 상담  
   이 세 가지 방식 모두 지원됩니다.  
   다만, 대면 상담은 거주 지역 및 제공 가능한 상담사에 따라 제한이 있을 수 있으므로 사전에 확인이 필요합니다.

3. **이용 시 주의사항 (provider 관련)**  
   - 상담 서비스는 Cigna가 지정한 **공인된 네트워크 내의 상담사 또는 전문가**를 통해서만 제공됩니다.  
   - 네트워크 외 상담사 이용 시 비용 보장이 제한될 수 있으므로, 상담 예약 전에 반드시 Cigna

### Demo 1-B: RAG 답변 (Cigna PDF 문서 기반)

# ── 디버그: 실제로 어떤 청크가 검색됐는지 확인 ──
print()
print('🔎 [검색된 청크 확인]')
retrieved_chunks = retriever.invoke(question)
for idx, chunk in enumerate(retrieved_chunks):
    page = chunk.metadata.get('page', 0)
    print(f'  [{idx+1}] p.{page+1}: {chunk.page_content[:120]}...')

In [ ]:
# ──────────────────────────────────────────────────────────────
# RAG 체인 실행: PDF에서 관련 청크 검색 → LLM 답변 생성
# ──────────────────────────────────────────────────────────────
print('=' * 65)
print('❓ 질문:', question)
print('=' * 65)
print()
print('📚 [RAG 답변] — Cigna 약관 PDF 검색 후 GPT-4.1-mini 응답')
print('-' * 65)

rag_response = rag_chain.invoke(question)
print(rag_response)
print()
print('✅ 위 답변은 Cigna Customer Guide PDF(p.36)에서 검색된 내용 기반입니다.')
print()
print('📌 [PDF 원문 확인]')
print('   Life Management Assistance Programme > Short-term counselling:')
print('   Up to 6 sessions via telephone, video, or face-to-face,')
print('   per issue per period of cover — Paid in full (Silver/Gold/Platinum)')

❓ 질문: Cigna 국제 의료보험(Global Health Options)에서 카운슬링(상담) 세션은 연간 몇 번까지 보장되나요? 어떤 방식(전화/화상/대면)으로 받을 수 있나요?

📚 [RAG 답변] — Cigna 약관 PDF 검색 후 GPT-4.1-mini 응답
-----------------------------------------------------------------
문서에서 카운슬링(상담) 세션의 연간 보장 횟수에 대한 구체적인 내용은 확인되지 않습니다.  
다만, 비응급(non-emergency) 건강 문제에 대해 전화 및 화상 상담은 Cigna WellbeingTM 앱이나 고객 서비스팀을 통한 의뢰로 무제한 제공됩니다.  
대면 상담에 관한 내용은 문서에서 확인되지 않습니다.

✅ 위 답변은 Cigna Customer Guide PDF(p.36)에서 검색된 내용 기반입니다.

📌 [PDF 원문 확인]
   Life Management Assistance Programme > Short-term counselling:
   Up to 6 sessions via telephone, video, or face-to-face,
   per issue per period of cover — Paid in full (Silver/Gold/Platinum)


---
## 📊 Demo 2: 표(Table) 기반 RAG — pdfplumber 활용

보험 약관 PDF에는 플랜별(Silver/Gold/Platinum) 보장 금액 **표**가 많습니다.  
일반 텍스트 추출로는 표 구조가 무너져 정확한 정보 검색이 어렵습니다.

**해결**: `pdfplumber`로 표를 구조적으로 추출 → 별도 벡터 저장소 구성  
*(Gemini 추천 방식: pdfplumber는 표 추출에 특히 효과적)*

In [ ]:
# ──────────────────────────────────────────────────────────────
# pdfplumber로 PDF 표 추출
# 참고: Gemini 추천 — 구조화된 표 데이터 추출에 pdfplumber가 효과적
# ──────────────────────────────────────────────────────────────
import pdfplumber

def extract_tables_from_pdf(pdf_path, pages=None):
    '''pdfplumber로 PDF에서 표를 추출하여 텍스트로 변환합니다.
    
    Args:
        pdf_path (str): PDF 파일 경로
        pages (list): 추출할 페이지 번호 리스트 (0-indexed). None이면 전체.
    
    Returns:
        list of dict: [{page, table_idx, table_text}, ...]
    '''
    table_texts = []
    
    with pdfplumber.open(pdf_path) as pdf:
        target_pages = pages if pages is not None else range(len(pdf.pages))
        
        for page_num in target_pages:
            if page_num >= len(pdf.pages):
                continue
            
            page = pdf.pages[page_num]
            tables = page.extract_tables()
            
            for table_idx, table in enumerate(tables):
                if not table or len(table) < 2:  # 헤더+데이터 최소 2행
                    continue
                
                # 각 행을 '|'로 구분된 텍스트로 변환
                lines = []
                for row in table:
                    cleaned = [
                        str(cell).strip().replace('\n', ' ') if cell else ''
                        for cell in row
                    ]
                    lines.append(' | '.join(cleaned))
                
                table_text = '\n'.join(lines)
                table_texts.append({
                    'page': page_num + 1,        # 1-indexed
                    'table_idx': table_idx + 1,
                    'table_text': table_text
                })
    
    return table_texts


# 주요 보장 내용이 있는 페이지 (0-indexed)
# p.16-18: 연간 최대, 병원비, 입원 현금, 응급, 재활 등
# p.36-38: 건강 & 웰빙, 카운슬링, 건강 검진 등
benefit_pages = list(range(15, 20)) + list(range(35, 39))

table_data = extract_tables_from_pdf(PDF_PATH, pages=benefit_pages)
print(f'📊 총 {len(table_data)}개 표 추출 완료')
print()
for item in table_data[:4]:
    print(f'--- Page {item["page"]}, 표 {item["table_idx"]} ---')
    print(item['table_text'][:250])
    print()

📊 총 30개 표 추출 완료

--- Page 16, 표 1 ---
Annual overall benefit maximum - per beneficiary per period of cover This includes claims paid across all sections of International Medical Insurance. | Silver |  | Gold |  | Platinum
 | $1,000,000 €800,000 £650,000 |  | $2,000,000 €1,600,000 £1,300,

--- Page 16, 표 2 ---
Hospital charges Up to the annual overall benefit maximum for your selected plan per beneficiary per period of cover. | Silver |  | Gold |  | Platinum
 | Paid in full Private room |  | Paid in full Private room |  | Paid in full Private room
› Nursin

--- Page 16, 표 3 ---
Hospital accommodation for a parent or guardian Up to the total limit shown for your selected plan per beneficiary per period of cover or, where “paid in full” is shown, this is up to the annual overall benefit maximum for your selected plan per bene

--- Page 17, 표 1 ---
Pandemics, epidemics and outbreaks of infectious illnesses Up to the annual overall benefit maximum for your selected plan per beneficiary per 

### 표 데이터 → LangChain Document → 벡터 저장소

추출한 표를 `Document` 객체로 변환하고, 페이지와 표 유형을 메타데이터로 저장합니다.  
*(출처: `02-8_metadata_filtering-da2a9e42.ipynb` — Document + metadata 패턴)*

In [32]:
# ──────────────────────────────────────────────────────────────
# 표 데이터 → Document 변환 (메타데이터 포함)
# 출처: 02-8_metadata_filtering-da2a9e42.ipynb — Document + metadata 패턴
# ──────────────────────────────────────────────────────────────
from langchain_core.documents import Document

table_docs = []
for item in table_data:
    doc = Document(
        page_content=(
            f'[Cigna 보험 혜택 표 | 페이지 {item["page"]}]\n'
            f'{item["table_text"]}'
        ),
        metadata={
            'source': 'Cigna_Customer_Guide_EN_2022',
            'page': item['page'],
            'type': 'benefit_table',
            'table_idx': item['table_idx']
        }
    )
    table_docs.append(doc)

print(f'✅ {len(table_docs)}개 표 Document 생성')

# 표 전용 벡터 저장소 생성
# 출처: 01_rag_workflow-9e1618d6.ipynb, Cell 20
table_vector_store = Chroma.from_documents(
    documents=table_docs,
    embedding=embedding_model,
    collection_name='cigna_benefit_tables'
)

table_retriever = table_vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

print(f'✅ 표 전용 벡터 저장소 생성: {table_vector_store._collection.count()}개 벡터')

✅ 30개 표 Document 생성
✅ 표 전용 벡터 저장소 생성: 30개 벡터


In [33]:
# ──────────────────────────────────────────────────────────────
# 표 기반 RAG 체인 구성
# 출처: 01_rag_workflow-9e1618d6.ipynb, Cell 30 (LCEL 체인 패턴)
# ──────────────────────────────────────────────────────────────
table_rag_chain = (
    {
        'question': RunnablePassthrough(),
        'context': table_retriever | format_docs
    }
    | rag_prompt
    | model
    | StrOutputParser()
)

print('✅ 표 기반 RAG 체인 구성 완료')

✅ 표 기반 RAG 체인 구성 완료


### 표 기반 Q&A 실행

In [50]:
# ──────────────────────────────────────────────────────────────
# 일반 LLM 직접 질문 (문서 검색 없음)
# ──────────────────────────────────────────────────────────────
from langchain_core.messages import HumanMessage

table_questions = [
    'Cigna Silver 플랜의 연간 최대 보장 금액(Annual overall benefit maximum)은 달러로 얼마인가요?',
    'Cigna Platinum 플랜에서 입원 현금 혜택(Inpatient cash benefit)은 1박에 얼마인가요?',
    'Cigna Gold 플랜의 재활 치료(Rehabilitation) 한도 금액과 최대 일수는 얼마인가요?'
]


# RAG 없이 직접 LLM에 질문
for i, q in enumerate(table_questions, 1):
    print(f'\n{"="*65}')
    print(f'📢 [일반 LLM 답변 {i}: {q}')
    print(f'{"="*65}')
    plain_response = model.invoke([HumanMessage(content=q)])
    print(plain_response.content)
    print('⚠️  주의: 위 답변은 LLM 학습 데이터 기반이며 실제 약관과 다를 수 있습니다.')
    print()




📢 [일반 LLM 답변 1: Cigna Silver 플랜의 연간 최대 보장 금액(Annual overall benefit maximum)은 달러로 얼마인가요?
Cigna Silver 플랜의 연간 최대 보장 금액(Annual overall benefit maximum)은 플랜 종류와 선택한 옵션에 따라 다를 수 있습니다. 일반적으로 Cigna의 Silver 건강보험 플랜은 연간 최대 보장 금액이 제한이 없거나 매우 높게 설정되어 있는 경우가 많습니다.

정확한 금액을 확인하려면 가입한 특정 플랜의 약관이나 Cigna 공식 웹사이트, 또는 고객 서비스에 문의하는 것이 가장 확실합니다. 만약 특정 플랜 문서나 요약 자료가 있다면, 그 문서 내 'Annual overall benefit maximum' 항목을 참고하시기 바랍니다.

필요하시면 플랜 세부 정보나 가입하신 지역, 플랜 ID 등을 알려주시면 더 구체적으로 도와드릴 수 있습니다.
⚠️  주의: 위 답변은 LLM 학습 데이터 기반이며 실제 약관과 다를 수 있습니다.


📢 [일반 LLM 답변 2: Cigna Platinum 플랜에서 입원 현금 혜택(Inpatient cash benefit)은 1박에 얼마인가요?
Cigna Platinum 플랜의 입원 현금 혜택(Inpatient cash benefit)은 플랜 종류와 국가에 따라 다를 수 있습니다. 일반적으로 Cigna Platinum 플랜에서는 1박당 약 50달러에서 100달러 사이의 현금 혜택을 제공하는 경우가 많습니다.

정확한 금액을 확인하시려면 가입하신 플랜의 보장 내용(Summary of Benefits)이나 보험 증권을 참고하시거나, Cigna 고객 서비스에 직접 문의하시는 것이 가장 정확합니다.
⚠️  주의: 위 답변은 LLM 학습 데이터 기반이며 실제 약관과 다를 수 있습니다.


📢 [일반 LLM 답변 3: Cigna Gold 플랜의 재활 치료(Rehabilitation) 한도 금액과 최대 일수는 얼마인가요?
Cigna Gold 플랜의 재활

In [34]:
# ──────────────────────────────────────────────────────────────
# Demo 2: pdfplumber로 추출한 표 데이터 기반 Q&A
# ──────────────────────────────────────────────────────────────
table_questions = [
    'Silver 플랜의 연간 최대 보장 금액(Annual overall benefit maximum)은 달러로 얼마인가요?',
    'Platinum 플랜에서 입원 현금 혜택(Inpatient cash benefit)은 1박에 얼마인가요?',
    'Gold 플랜의 재활 치료(Rehabilitation) 한도 금액과 최대 일수는 얼마인가요?'
]

for i, q in enumerate(table_questions, 1):
    print(f'\n{"="*65}')
    print(f'❓ 표 기반 질문 {i}: {q}')
    print(f'{"="*65}')
    answer = table_rag_chain.invoke(q)
    print(answer)


❓ 표 기반 질문 1: Silver 플랜의 연간 최대 보장 금액(Annual overall benefit maximum)은 달러로 얼마인가요?
Silver 플랜의 연간 최대 보장 금액(Annual overall benefit maximum)은 $1,000,000입니다.

❓ 표 기반 질문 2: Platinum 플랜에서 입원 현금 혜택(Inpatient cash benefit)은 1박에 얼마인가요?
Platinum 플랜에서 입원 현금 혜택(Inpatient cash benefit)은 1박에 $200, €150, £130입니다.

❓ 표 기반 질문 3: Gold 플랜의 재활 치료(Rehabilitation) 한도 금액과 최대 일수는 얼마인가요?
Gold 플랜의 재활 치료 한도는 $10,000 (또는 €7,400, £6,650)이며, 최대 60일입니다.


---
## 📝 정리 및 다음 단계

### Demo 결과 요약
| | 일반 LLM | RAG (Demo 1) | 표 RAG (Demo 2) |
|---|---|---|---|
| 데이터 출처 | 학습 데이터 | Cigna PDF 텍스트 | Cigna PDF 표 |
| 정확도 | 낮음 (불확실) | 높음 | 매우 높음 (수치) |
| 환각(Hallucination) | 발생 가능 | 억제됨 | 억제됨 |

### 향후 개선 방향
1. **다국어 처리** — BAAI/bge-m3 임베딩으로 한국어 질문 → 영어 문서 검색
2. **용어 고정(Term Locking)** — 보험 전문 용어 번역 일관성 확보
3. **파인튜닝** — Qwen2.5-7B + QLoRA로 보험 도메인 특화 모델 학습
4. **메타데이터 필터링** — 플랜 유형(Silver/Gold/Platinum) 필터 추가

---
## 🤖 Demo 3: Qwen2.5-7B-Instruct — 로컬 LLM 기반 RAG

OpenAI API 없이 **오픈소스 LLM(Qwen2.5-7B-Instruct)**으로 동일한 RAG 파이프라인을 구동합니다.

| | Demo 1-2 (OpenAI) | Demo 3 (Qwen) |
|---|---|---|
| LLM | gpt-4.1-mini (API) | Qwen2.5-7B-Instruct (로컬) |
| 비용 | API 과금 | 무료 (GPU VRAM ~10GB) |
| 속도 | 빠름 | GPU 환경 권장 |
| 파인튜닝 | 불가 | QLoRA 파인튜닝 가능 ← 차기 계획 |

> **환경 권장**: GPU VRAM 10GB 이상 (Colab T4 / RTX 3090 등)  
> CPU만 있는 경우 `load_in_4bit=True` 옵션으로 속도는 느리지만 동작 가능

In [ ]:
# ──────────────────────────────────────────────────────────────
# Demo 3 - Step 1: 패키지 설치
# ──────────────────────────────────────────────────────────────
# GPU 환경 (Colab / 로컬 CUDA)
!pip install transformers accelerate bitsandbytes torch --quiet

# CPU 전용 환경 (느리지만 동작)
# !pip install transformers accelerate torch --quiet

print("✅ 패키지 설치 완료")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Demo 3 - Step 2: Qwen2.5-7B-Instruct 다운로드 & 로드
# ──────────────────────────────────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

print(f"⏬ 모델 다운로드 중: {MODEL_NAME}")
print("   첫 실행 시 약 15GB 다운로드 — 시간이 걸립니다 (캐시되면 이후 빠름)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",         # GPU 있으면 자동 할당, 없으면 CPU
    load_in_4bit=True,         # 4-bit 양자화 → VRAM ~10GB로 절감
)

device = next(model.parameters()).device
print(f"\n✅ 모델 로드 완료")
print(f"   디바이스: {device}")
print(f"   dtype   : {next(model.parameters()).dtype}")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Demo 3 - Step 3: 기본 추론 함수 정의
# ──────────────────────────────────────────────────────────────
def ask_qwen(user_message: str, system_message: str = None, max_new_tokens: int = 512) -> str:
    if system_message is None:
        system_message = (
            "You are a helpful assistant specializing in international health insurance. "
            "Answer based only on the provided documents. "
            "Always cite the source page at the end of your answer."
        )

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user",   "content": user_message},
    ]

    # Qwen2.5 공식 chat template 적용
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # 입력 토큰 제거 → 생성된 토큰만 디코딩
    generated = output_ids[0][len(inputs.input_ids[0]):]
    return tokenizer.decode(generated, skip_special_tokens=True)


# ── 빠른 동작 테스트 ──
test_answer = ask_qwen("What is a deductible in health insurance?", max_new_tokens=128)
print("🧪 동작 테스트")
print(f"Q: What is a deductible in health insurance?")
print(f"A: {test_answer}")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Demo 3 - Step 4: Qwen2.5-7B + Cigna RAG 통합
# retriever 는 앞서 Demo 1에서 생성한 객체를 그대로 사용
# ──────────────────────────────────────────────────────────────

SYSTEM_PROMPT_KO = """당신은 Cigna 국제 의료보험 약관 전문 상담 어시스턴트입니다.
아래 [참고 문서] 내용만을 근거로 답변하세요.
- 문서에 없는 내용은 "제공된 문서에서 확인할 수 없습니다"라고 답하세요.
- 보험 추천이나 법적 판단은 하지 않으며, 정보 제공만 합니다.
- 답변 끝에 반드시 출처 페이지를 명시하세요.
- 한국어로 답변하세요."""


def qwen_rag_answer(question: str):
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")

    # 1. VectorDB 검색 (기존 retriever 재사용)
    retrieved_docs = retriever.invoke(question)
    context_parts = []
    for i, doc in enumerate(retrieved_docs):
        page = doc.metadata.get('page', '?')
        page_display = page + 1 if isinstance(page, int) else page
        context_parts.append(f"[문서 {i+1} | p.{page_display}]\n{doc.page_content}")
    context = "\n\n".join(context_parts)

    # 2. 프롬프트 구성
    user_message = f"""[참고 문서]
{context}

[질문]
{question}"""

    # 3. Qwen 추론
    answer = ask_qwen(user_message, system_message=SYSTEM_PROMPT_KO, max_new_tokens=512)

    print(f"\nA (Qwen2.5-7B): {answer}")
    print(f"\n📄 검색된 청크 {len(retrieved_docs)}개:")
    for i, doc in enumerate(retrieved_docs):
        page = doc.metadata.get('page', '?')
        page_display = page + 1 if isinstance(page, int) else page
        print(f"  [{i+1}] p.{page_display}: {doc.page_content[:80]}...")
    return answer


# ── Demo 3 실행 ──
qwen_rag_answer(
    "Life Management Assistance Programme의 counselling 세션은 몇 회까지 커버됩니까? "
    "전화, 화상, 대면 중 어떤 방식이 가능하고, 주의사항은 무엇입니까?"
)

### Demo 3 결과 해석

| 항목 | 내용 |
|------|------|
| **모델** | Qwen2.5-7B-Instruct (4-bit 양자화) |
| **추론 방식** | RAG — Cigna PDF 청크 검색 후 Qwen이 한국어 답변 생성 |
| **다음 단계** | QLoRA 파인튜닝으로 면책 고지 자동 삽입 · 보험 추천 거절 · 용어 고정 학습 |

> 파인튜닝 계획은 `3차_프로젝트_기획_정리.md` 섹션 7 참고